In [2]:
import numpy as np

X_train = np.load(r"E:\Hybrid_IDS_Project\dataset\processed\x_train_scaled.npy")
X_test = np.load(r"E:\Hybrid_IDS_Project\dataset\processed\x_test_scaled.npy")

y_train = np.load(r"E:\Hybrid_IDS_Project\dataset\processed\y_train.npy")
y_test = np.load(r"E:\Hybrid_IDS_Project\dataset\processed\y_test.npy")

print(X_train.shape)
print(y_train.shape)

(2016638, 78)
(2016638,)


In [3]:
X_train_normal = X_train[y_train == 0]

print(X_train_normal.shape)

(1676045, 78)


In [4]:
from sklearn.svm import OneClassSVM

In [5]:

ocsvm = OneClassSVM(
    kernel="rbf",
    gamma="scale",
    nu=0.05
)

In [6]:
from sklearn.utils import resample

# Sample 50,000 normal samples
X_train_normal_sample = resample(
    X_train_normal,
    n_samples=50000,
    random_state=42
)

print(X_train_normal_sample.shape)

(50000, 78)


In [7]:
ocsvm.fit(X_train_normal_sample)

,"nu nu: float, default=0.5An upper bound on the fraction of trainingerrors and a lower bound of the fraction of supportvectors. Should be in the interval (0, 1]. By default 0.5will be taken.",0.05
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm.If none is given, 'rbf' will be used. If a callable is given it isused to precompute the kernel matrix.",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide <shrinking_svm>`.",True
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False
,"max_iter max_iter: int, default=-1Hard limit on iterations within solver, or -1 for no limit.",-1
Name,Type,Value


In [8]:
print(X_train_normal.shape)

(1676045, 78)


In [9]:
y_pred = ocsvm.predict(X_test)

In [10]:
import numpy as np

# Convert OCSVM output to binary labels
# Normal (1) -> 0
# Anomaly (-1) -> 1

y_pred = np.where(y_pred == 1, 0, 1)

In [11]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)

print(classification_report(
    y_test,
    y_pred,
    target_names=["BENIGN", "ATTACK"]
))

Accuracy : 0.8880295937797524
Precision: 0.7012214789571851
Recall   : 0.5872363414290412
F1 Score : 0.6391869866734844
              precision    recall  f1-score   support

      BENIGN       0.92      0.95      0.93    419012
      ATTACK       0.70      0.59      0.64     85148

    accuracy                           0.89    504160
   macro avg       0.81      0.77      0.79    504160
weighted avg       0.88      0.89      0.88    504160



In [12]:
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

fpr = fp / (fp + tn)

print("TN:", tn)
print("FP:", fp)
print("FN:", fn)
print("TP:", tp)
print("False Positive Rate:", fpr)

TN: 397707
FP: 21305
FN: 35146
TP: 50002
False Positive Rate: 0.050845799165656354


In [13]:
import joblib

joblib.dump(
    ocsvm,
    r"E:\Hybrid_IDS_Project\dataset\models\ocsvm.pkl"
)

['E:\\Hybrid_IDS_Project\\dataset\\models\\ocsvm.pkl']

In [14]:
import pandas as pd

results = pd.read_csv(
    r"E:\Hybrid_IDS_Project\dataset\results\model_results.csv"
)

ocsvm_result = pd.DataFrame({
    "Model": ["OCSVM"],
    "Experiment": ["Known Attack - Binary"],
    "Accuracy": [accuracy],
    "Precision": [precision],
    "Recall": [recall],
    "F1_Score": [f1],
    "FPR": [fpr],
    "TN": [tn],
    "FP": [fp],
    "FN": [fn],
    "TP": [tp]
})

results = pd.concat([results, ocsvm_result], ignore_index=True)

results.to_csv(
    r"E:\Hybrid_IDS_Project\dataset\results\model_results.csv",
    index=False
)

print(results)

           Model             Experiment  Accuracy  Precision    Recall  \
0  Random Forest  Known Attack - Binary  0.998929   0.994483  0.999201   
1            CNN  Known Attack - Binary  0.984414   0.921080  0.992777   
2          OCSVM  Known Attack - Binary  0.888030   0.701221  0.587236   

   F1_Score       FPR      TN     FP     FN     TP  
0  0.996837  0.001126  418540    472     68  85080  
1  0.955585  0.017286  411769   7243    615  84533  
2  0.639187  0.050846  397707  21305  35146  50002  
